# Cleared for Departure? Flight Delays and Cancellations in Five U.S. Regions

**Team:** Texas  
**Course:** INFO 450  
**Team members:**
Nathnael Abraham
Andrea Atoa Ortiz
Nahom Ghiorghis
Danny Truong

## Phase 1 — Business Understanding

### Research questions
1. How reliable is flying out of DFW, IAH, AUS, and SAT when comparing on-time arrival percentage and cancellation rate?
2. How concentrated are arrival delay minutes across routes and individual flights?
3. When are delays worst across month, day of week, and scheduled departure hour?
4. Which airlines and carrier types are most and least reliable in the Texas sample?

### Audience
Airport authorities, airline operations teams, corporate travel managers, and travelers choosing between Texas airports could use these results to compare reliability and identify periods or routes with more disruption.

### Expectations before analysis
We expect the busiest airports to have more total delays because they handle more flights. We also expect delay levels to change by season and time of day, and we expect a relatively small group of flights or routes to account for a large share of delay minutes.

## Phase 2 — Data Understanding

In [2]:
from src.data_profile import load_data, profile_data, missingness_evidence, show_missingness_checks

raw = load_data()
missing_table = profile_data(raw)
missingness_table = missingness_evidence(raw)
display(missingness_table)
show_missingness_checks(raw)

ModuleNotFoundError: No module named 'src'

### Redundant or uninformative columns

The columns dropped in Phase 3 are `Year`, `Quarter`, `DayofMonth`, `DOT_ID_Reporting_Airline`, `OriginAirportID`, `OriginStateFips`, `DepTimeBlk`, and `Flights`. The date and airline/airport codes retain the information needed for analysis, while the time block can be derived from scheduled departure time and each row represents one flight. `Month`, `DayOfWeek`, and `ArrDel15` are retained because the assignment specifically uses them later. `DistanceGroup` is retained until missing `Distance` values have been assessed.

### Missing-data classification

- `CancellationCode`: structural for non-cancelled flights because a cancellation reason does not exist for those records.
- `ArrDelay`, `ArrDel15`, and `ActualElapsedTime`: structural for cancelled and diverted flights because they did not complete the scheduled arrival.
- Delay-cause columns: structural when `ArrDel15` is 0 because the assignment says causes are reported only for flights delayed at least 15 minutes.
- Departure fields and `TaxiOut`: mostly explained by cancellations and diversions; missingness is consistent with structural absence when a flight did not depart normally. A small number of missing values among non-cancelled flights should be checked separately.
- `Distance`: 600 missing values are not explained by cancellation or diversion. `DistanceGroup` remains available as a broad distance category, but it cannot recover the exact mileage. Leave exact distance missing for distance-specific analysis.
- `Tail_Number`: missing values occur on cancelled flights, consistent with the aircraft not being assigned or recorded.
- `CancellationCode`: missing on non-cancelled flights is not an error and should not be filled.

These are evidence-based classifications, not proof of the underlying missingness mechanism.

## Phase 3 — Data Preparation

In [ ]:
from src.prepare_data import prepare_data

clean = prepare_data()
display(clean.head())
display(clean[["FlightDate", "DepHour", "Weekend", "Route", "DelayCategory", "DelayMinutes"]].head())

### Cleaning decisions

Exact duplicate rows are removed, keeping one copy. Carrier and destination lookup data are merged using their airport/airline codes. `FlightDate` is converted to datetime, and the required derived columns are created. The cleaned file keeps all scheduled flights, including cancelled and diverted flights.

Missing values are not filled indiscriminately. Arrival-delay statistics use completed flights only (`Cancelled = 0` and `Diverted = 0`). Cancellation and diversion rates use all scheduled flights. Distance remains missing where the exact value is unavailable. Delay-cause values remain blank when a flight was not at least 15 minutes late.

### Duplicate Records and Data Quality

The original Texas flight dataset contained 60000 rows. After checking for exact duplicate records across all columns, we identified 180 duplicate rows. These records were identical to other rows in the dataset, meaning they represented repeated copies rather than separate scheduled flights.

We removed the extra copies while keeping one instance of each unique record. This prevents the same flight from being counted more than once and helps ensure that our flight totals, percentages, and other statistical results are accurate. After removing duplicates, the dataset contained 59,820 rows.

## Phase 4 — Early Analysis

In [ ]:
from src.airport_analysis import airport_reliability, arrival_delay_summary, airport_charts
from src.time_analysis import monthly_trend, daily_rolling_trend

airport_results = airport_reliability(clean)
arrival_delay_summary(clean)
airport_charts(airport_results, clean)
monthly_results = monthly_trend(clean)
daily_results = daily_rolling_trend(clean)

### Preliminary interpretation

Use the airport table and charts to describe the early differences in on-time performance and cancellations. The monthly and daily charts are exploratory because the dataset is a random sample of departures, so daily rates can be noisy. More analysis is still needed for the concentration measures, full inferential statistics, selected research questions, and evaluation.

## AI Use Disclosure

We used AI to help clean/organize our project, clean/improve draft code and grammatical errors, and troubleshoot any mismatching values/id variables we used since we coded separately.